# Notebook 2 . Hapus Kolom

Isi `Input_CSV` dengan lokasi file yang ingin diproses. Atur `Pertahankan_Kolom` untuk memilih kolom yang disimpan, atau `Hapus_Kolom` untuk membuang kolom tertentu. File sumber tidak ditimpa. Semua kolom dibaca sebagai teks agar ID seperti `00123` tetap utuh.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Input_CSV = Path("Input/input.csv")  # Ganti dengan lokasi CSV yang akan diproses
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma

# Pilih salah satu cara di bawah ini:
# Pertahankan_Kolom: isi jika hanya kolom tertentu yang ingin disimpan.
Pertahankan_Kolom = None  # Contoh: ["id", "nama", "kota"]; isi None jika ingin memakai Hapus_Kolom
# Hapus_Kolom: isi kolom yang ingin dibuang jika Pertahankan_Kolom = None.
Hapus_Kolom = ["kolom_lama", "catatan_internal"]  # Contoh: ["kolom_lama", "catatan_internal"]

Hapus_Kolom_Kosong = True  # Buang kolom yang seluruh nilainya kosong
Rapikan_Spasi = True
Output_CSV = Path("Output/hasil_hapus_kolom.csv")
Encoding_Output = "utf-8-sig"

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Baca dan periksa kolom

In [ ]:
input_path = Input_CSV if Input_CSV.is_absolute() else BASE_DIR / Input_CSV
if not input_path.exists():
    raise FileNotFoundError(f"File tidak ditemukan: {input_path}")

df = pd.read_csv(input_path, sep=Separator_CSV, dtype="string", encoding="utf-8-sig")
df.columns = df.columns.astype(str).str.strip()
if Rapikan_Spasi:
    for col in df.select_dtypes(include=["string", "object"]).columns:
        df[col] = df[col].str.strip()

print(f"File: {input_path}")
print(f"Ukuran awal: {len(df):,} baris x {len(df.columns)} kolom")
print("Kolom tersedia:")
for col in df.columns:
    print(f"- {col}")
display(df.head())


## Atur kolom hasil

Jika `Pertahankan_Kolom` diisi daftar kolom, hanya kolom tersebut yang disimpan. Jika bernilai `None`, kolom pada `Hapus_Kolom` yang dibuang.

In [ ]:
if Pertahankan_Kolom is not None:
    missing = [col for col in Pertahankan_Kolom if col not in df.columns]
    if missing:
        raise KeyError(f"Kolom Pertahankan_Kolom tidak ditemukan: {missing}")
    result = df.loc[:, Pertahankan_Kolom].copy()
else:
    missing = [col for col in Hapus_Kolom if col not in df.columns]
    if missing:
        print(f"Kolom Hapus_Kolom tidak ditemukan (dilewati): {missing}")
    kolom_dihapus = [col for col in Hapus_Kolom if col in df.columns]
    result = df.drop(columns=kolom_dihapus).copy()

if Hapus_Kolom_Kosong:
    empty_cols = [col for col in result.columns if result[col].isna().all()]
    result = result.drop(columns=empty_cols)
else:
    empty_cols = []

removed = [col for col in df.columns if col not in result.columns]
print(f"Kolom dibuang: {removed if removed else 'tidak ada'}")
print(f"Ukuran hasil: {len(result):,} baris x {len(result.columns)} kolom")
display(result.head())


## Simpan hasil

In [ ]:
output_path = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False, encoding=Encoding_Output, sep=Separator_CSV)
print(f"Tersimpan: {output_path.resolve()}")